# DS207 Final Project - Predicting Readmission Rate for Diabetic Patients Using Machine Learning
#### Contributor: Rahil Sharma (rahilsharma@berkeley.edu)

### Notebook Structure:

1. Data Processing:
2. Baseline Model: 
    * Majority
    * Multiclass Logistic Regression
3. Notebook Exports:
    * Datasets: `results/train.csv`, `results/val.csv`, and `results/test.csv`
    * Baseline model: in `baseline.pkl`

## 1. Data Processing

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib



2025-12-07 20:16:10.882957: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-07 20:16:10.886250: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-07 20:16:10.901872: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-07 20:16:10.924819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-07 20:16:10.945892: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [2]:
df = pd.read_csv('../data/diabetic_data.csv')
raw_df = pd.read_csv('../data/diabetic_data.csv')
display(df.head())

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### Key for Drug descriptions ##

Up: The dosage of the drug was increased for the patient during their encounter.

Down: The dosage was decreased.

Steady: The patient is on the drug, and the dosage was not changed.

No: The patient was not prescribed this specific drug.

In [3]:
print(f"There are {len(df.columns)} columns:\n", df.columns)

There are 50 columns:
 Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')


#### Drop duplicates and unnecessary columns

In [4]:
df = df.drop_duplicates(subset=['patient_nbr'], keep='first')

In [5]:
# checking to see how many missing values there are
(df == '?').sum()

# columns to keep based on descriptions / number of missing values
  # race (split into indicator variables)
  # gender (split into indicator variables)
  # age (split into indicator variables)
  # time_in_hospital (int)
  # num_lab_procedures (int)
  # num_procedures (int)
  # num_medications (int)
  # number_outpatient (int)
  # number_emergency (int)
  # number_inpatient (int)
  # number_diagnoses (int)
  # keeping all the indicator variables for medication / medication changes
  # target - multiclass classification

# dropping the unnecessary columns
df = df.drop(columns=['encounter_id', 'patient_nbr', 'weight', 'admission_type_id',
                      'discharge_disposition_id', 'admission_source_id', 'payer_code',
                      'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
                      'A1Cresult'],
             axis=1)

#### Collapse age and race

In [6]:
# Convert age buckets like "[70-80)" to numeric midpoint (75)
def age_to_mid(age_bucket):
    if pd.isna(age_bucket):
        return np.nan
    try:
        lo, hi = age_bucket.strip('[]').split('-')
        lo = int(lo)
        hi = int(hi.strip(')'))
        return (lo + hi) / 2
    except Exception:
        return np.nan
df['age_mid'] = df['age'].apply(age_to_mid)
df.drop(columns=['age'], inplace=True)

In [7]:
# Collapse low count race categories into 'Other'
if 'race' in df.columns:
    df['race'] = df['race'].fillna('Unknown')
    top = df['race'].value_counts().nlargest(4).index
    df['race_collapsed'] = df['race'].where(df['race'].isin(top), other='Other')
    df.drop(columns=['race'], inplace=True)

#### Categorical Features

In [8]:
# creating indicator variables for the categorical features (one-hot encoding)
df = pd.get_dummies(df, columns=['race_collapsed', 'gender'], dtype=int)

# Encode ordinal values according to the prescription status of the visit
prescription_map = {'No': 0,       # The drug was not prescribed
                    'Down': 1,     # The dosage was decreased
                    'Steady': 2,   # The dosage did not change
                    'Up': 3}       # The dosage was increased during the encounter
cat_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
               'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
               'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
               'miglitol', 'troglitazone', 'tolazamide', 'examide',
               'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
               'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
for cat_col in cat_columns:
    df[cat_col] = df[cat_col].map(prescription_map)

#### Binary features and Outcome of Interest

In [9]:
# displaying the data
print("The number of columns are: ", len(df.columns))
display(df.head())
print(df.columns)

# recoding change and diabetesMed to 0 for No and 1 to Yes
df['change'] = np.where(df['change'] == 'No', 0, 1)
df['diabetesMed'] = np.where(df['diabetesMed'] == 'No', 0, 1)

# mapping the target into three classes
readmission_map = {
    'NO': 0,  # no readmission recorded
    '>30': 0, # readmitted (over 30 days)
    '<30': 1  # readmitted (within 30 days)
}

df['readmitted'] = df['readmitted'].map(readmission_map)

display(df.head())

df['readmitted'].value_counts()


The number of columns are:  43


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,NO,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,>30,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,NO,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,NO,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,NO,45.0,0,0,1,0,0,0,1,0


Index(['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed', 'readmitted', 'age_mid', 'race_collapsed_?',
       'race_collapsed_AfricanAmerican', 'race_collapsed_Caucasian',
       'race_collapsed_Hispanic', 'race_collapsed_Other', 'gender_Female',
       'gender_Male', 'gender_Unknown/Invalid'],
      dtype='object')


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,0,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,0,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,0,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,0,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,0,45.0,0,0,1,0,0,0,1,0


readmitted
0    65225
1     6293
Name: count, dtype: int64

In [10]:
df.describe()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
count,71518.00000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,...,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000,71518.000000
mean,4.28913,43.075478,1.430577,15.705025,0.280069,0.103540,0.177829,7.245700,0.424858,0.026511,...,0.087992,65.651864,0.027238,0.180192,0.747938,0.021211,0.023421,0.531684,0.468274,0.000042
std,2.94921,19.952338,1.759864,8.311163,1.068957,0.509187,0.603790,1.994674,0.835638,0.234470,...,0.283285,15.978075,0.162777,0.384350,0.434200,0.144090,0.151236,0.498999,0.498996,0.006477
min,1.00000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.00000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,...,0.000000,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.00000,44.000000,1.000000,14.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,...,0.000000,65.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,6.00000,57.000000,2.000000,20.000000,0.000000,0.000000,0.000000,9.000000,0.000000,0.000000,...,0.000000,75.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,14.00000,132.000000,6.000000,81.000000,42.000000,42.000000,12.000000,16.000000,3.000000,3.000000,...,1.000000,95.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### Shuffling and splitting

In [11]:
# randomly shuffling the data
display(df.head())

indices = np.arange(len(df))

shuffled_indices = np.random.permutation(indices)

df = df.iloc[shuffled_indices].reset_index(drop=True)

display(df.head())

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,1,41,0,1,0,0,0,1,0,0,...,0,5.0,0,0,1,0,0,1,0,0
1,3,59,0,18,0,0,0,9,0,0,...,0,15.0,0,0,1,0,0,1,0,0
2,2,11,5,13,2,0,1,6,0,0,...,0,25.0,0,1,0,0,0,1,0,0
3,2,44,1,16,0,0,0,7,0,0,...,0,35.0,0,0,1,0,0,0,1,0
4,1,51,0,8,0,0,0,5,0,0,...,0,45.0,0,0,1,0,0,0,1,0


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,readmitted,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
0,3,42,1,21,0,0,0,7,0,0,...,0,75.0,0,0,1,0,0,1,0,0
1,3,21,2,16,9,0,0,9,0,0,...,0,35.0,0,0,1,0,0,1,0,0
2,1,19,6,14,0,0,0,7,0,0,...,0,75.0,0,0,1,0,0,0,1,0
3,2,49,3,16,7,0,0,9,0,0,...,0,65.0,0,0,1,0,0,1,0,0
4,3,45,5,20,0,0,0,9,0,0,...,0,45.0,0,0,1,0,0,0,1,0


In [12]:
# splitting X and Y data
X = df.copy().drop(columns=['readmitted'], axis=1)
Y = df.copy()['readmitted']

In [13]:
# splitting the data into train, val, test (60/20/20)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, random_state=1234)
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, train_size=0.75, random_state=1234)

#### Continuous feature standardizations

In [14]:
# standardizing the continous features between 0 and 1
columns_to_standardize = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
                          'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_mid']

scaler = MinMaxScaler()

X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
X_val[columns_to_standardize] = scaler.transform(X_val[columns_to_standardize])
X_test[columns_to_standardize] = scaler.transform(X_test[columns_to_standardize])

print(f"The shape of X_train is {X_train.shape}")
print(f"\nThe shape of Y_train is {Y_train.shape}")
print(f"\nThe shape of X_val is {X_val.shape}")
print(f"\nThe shape of Y_val is {Y_val.shape}")
print(f"\nThe shape of X_test is {X_test.shape}")
print(f"\nThe shape of Y_test is {Y_test.shape}")


The shape of X_train is (42910, 42)

The shape of Y_train is (42910,)

The shape of X_val is (14304, 42)

The shape of Y_val is (14304,)

The shape of X_test is (14304, 42)

The shape of Y_test is (14304,)


In [15]:
display(X_train.head())
display(Y_train.head())

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,...,diabetesMed,age_mid,race_collapsed_?,race_collapsed_AfricanAmerican,race_collapsed_Caucasian,race_collapsed_Hispanic,race_collapsed_Other,gender_Female,gender_Male,gender_Unknown/Invalid
14839,0.846154,0.427481,0.666667,0.2875,0.0,0.00000,0.0,0.533333,0,0,...,1,0.666667,0,1,0,0,0,1,0,0
69258,0.000000,0.137405,0.333333,0.1375,0.0,0.02381,0.0,0.466667,0,0,...,1,0.777778,0,0,1,0,0,0,1,0
447,0.076923,0.381679,0.000000,0.1500,0.0,0.00000,0.0,0.533333,0,0,...,1,0.888889,1,0,0,0,0,0,1,0
9126,0.153846,0.160305,0.500000,0.0500,0.0,0.00000,0.0,0.533333,0,0,...,1,0.888889,0,0,1,0,0,0,1,0
46337,0.615385,0.343511,1.000000,0.3000,0.0,0.00000,0.0,0.533333,0,0,...,1,0.777778,0,0,1,0,0,1,0,0


14839    0
69258    0
447      0
9126     0
46337    0
Name: readmitted, dtype: int64

## 2. Baseline Model Developments

### 2.1 Multiclass logistic regression (baseline model)

In [16]:
def build_model(learning_rate=0.01):

  tf.keras.backend.clear_session()

  model = tf.keras.Sequential()

  model.add(keras.Input(shape=(X_train.shape[1],)))

  model.add(keras.layers.Dense(
      units=64,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=32,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=16,
      activation='relu'
  ))

  # model.add(keras.layers.BatchNormalization())
  # model.add(keras.layers.Dropout(0.2))

  model.add(keras.layers.Dense(
      units=1,
      activation='sigmoid',
      kernel_initializer='glorot_uniform',
      bias_initializer='glorot_uniform'
  ))

  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
      loss=keras.losses.BinaryCrossentropy(),
      metrics=['accuracy','Precision', 'Recall', 'AUC', 'f1_score']
  )

  history = model.fit(
      x=X_train,
      y=Y_train,
      validation_data=(X_val, Y_val),
      batch_size=64,
      epochs=10,
      verbose=1
  )

  return model, history

In [17]:
m1, history = build_model(0.001)

Epoch 1/10


I0000 00:00:1765167375.171621  126475 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-12-07 20:16:15.172238: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


671/671 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - AUC: 0.5202 - Precision: 0.1121 - Recall: 0.0034 - accuracy: 0.9096 - f1_score: 0.1622 - loss: 0.3111 - val_AUC: 0.5465 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.9119 - val_f1_score: 0.1619 - val_loss: 0.2977
Epoch 2/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.5418 - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9117 - f1_score: 0.1622 - loss: 0.2996 - val_AUC: 0.5524 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.9119 - val_f1_score: 0.1619 - val_loss: 0.2971
Epoch 3/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.5565 - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9117 - f1_score: 0.1622 - loss: 0.2982 - val_AUC: 0.5598 - val_Precision: 0.0000e+00 - val_Recall: 0.0000e+00 - val_accuracy: 0.9119 - val_f1_score: 0.1619 - val_loss: 0.2964
Epoch 4/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - AUC: 0.5752 - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accura

In [18]:
preds = (m1.predict(X_val) > 0.5).astype(int)
print(np.unique(preds, return_counts=True))

447/447 ━━━━━━━━━━━━━━━━━━━━ 0s 711us/step
(array([0, 1]), array([14300,     4]))


## 3. Notebook Exports

#### 3.1 Export the training, validation, and testing datasets

In [19]:
df_train = pd.concat([X_train, Y_train], axis=1).to_csv('../results/train.csv', index=False)
df_val = pd.concat([X_val, Y_val], axis=1).to_csv('../results/val.csv', index=False)
df_test = pd.concat([X_test, Y_test], axis=1).to_csv('../results/test.csv', index=False)

#### 3.2 Export baseline model(s)

In [20]:
joblib.dump(m1, '../results/baseline.pkl')

['../results/baseline.pkl']